In [1]:
import numpy as np
import pandas as pd

In [ ]:
factor = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/sample_oot.parquet')
daily_corr = factor.groupby('trade_date').apply(lambda x: x[factor.columns[8:]].corrwith(x['Y_20d'])).reset_index()

In [ ]:
"""
t: pd.Timestamp, time
Style: pd.DataFrame, Style factor explosure
都是只有一天的数据
"""
# 风格因子暴露，已经做了标准化
Style = Style.droplevel(1).sort_index()
# 行业哑变量矩阵，采用申万一级行业分类
ind = get_industry(date=t, industry_type=['sw_l1'])
ind['cons'] = 1
Industry = ind.pivot_table(index='code', columns='industry_name', values='cons').fillna(0.)
Industry, Style = Industry.align(Style, join='inner', axis=0)
# 国家因子暴露，全为1
Country = pd.Series(1., index=Style.index, name='Country')
# 合并
X = pd.concat([Country, Industry, Style], axis=1)
# 异方差调整，WLS权重计算
val = get_valuation(end_date=t, count=1, fields=['circulating_market_cap']).set_index('code')['circulating_market_cap']
V = val.align(X, axis=0, join='right')[0].fillna(0.)
V = pd.DataFrame(np.diag(np.sqrt(V) / np.sum(np.sqrt(V))), index=V.index, columns=V.index)
ind['Size'] = ind['code'].map(val)
industry_weights = ind.groupby('industry_name')['Size'].sum() / ind['Size'].sum()
industry_weights = industry_weights[Industry.columns]


In [ ]:
# 约束矩阵R计算
k = len(X.columns)
diag_R = np.diag(np.ones(k))
location = len(industry_weights)
R = np.delete(diag_R, location, axis=1)
adj_industry_weights = -industry_weights.div(industry_weights.iloc[-1]).iloc[:-1]
R[location, 1:location] = adj_industry_weights.values
# 因子权重计算
W = R@np.linalg.pinv(R.T@X.T@V@X@R)@R.T@X.T@V
W.index = X.columns
# 纯因子收益率计算
price = get_price(start_date=t+pd.Timedelta(days=1), count=1, fields=['pre_close', 'close'])
price['r'] = price.eval('close/pre_close - 1')
r = price.set_index('code')['r']
r = r.align(X, join='right')[0].fillna(0.)
factor_return = W.dot(r).to_frame(name=t).T